# Chicken-threat candidate training
Run this launcher in Kaggle with the approved dataset and fixed evaluation inputs attached.

In [1]:
!find /kaggle/input -maxdepth 3 -type f | head -50
!find /kaggle/input -name data.yaml -print
!nvidia-smi

/kaggle/input/datasets/dumdart/chicken-thread-detector-v5-0-0/data.yaml
Tue Jul 14 11:24:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |

In [2]:
REPOSITORY_URL = "https://github.com/Dumdart/SmartHomeBridge.git"
COMMIT_SHA = "1bf2b02ed23f00511346a2041330acb39ec9e53a"

V4_TEST_YAML = "/kaggle/input/chicken-threat-v4-test/data.yaml"
BARN_HOLDOUT_YAML = "/kaggle/input/chicken-threat-barn-holdout/data.yaml"
DATASET_MANIFEST = "/kaggle/input/YOUR-DATASET-SLUG/dataset_manifest.json"

DATASET_YAML = "/kaggle/input/datasets/dumdart/chicken-thread-detector-v5-0-0/data.yaml"
DATASET_MANIFEST = "/kaggle/input/datasets/dumdart/chicken-thread-detector-v5-0-0/dataset_manifest.json" 

OUTPUT_DIR = "/kaggle/working/chicken-threat-candidate"
MODEL_ID = "chicken-threat-yolo11m-v5.0.0"

In [3]:
from pathlib import Path
import yaml

DATASET_ROOT = Path(
    "/kaggle/input/datasets/dumdart/chicken-thread-detector-v5-0-0"
)

for split in ("train", "valid", "test"):
    images = DATASET_ROOT / split / "images"
    labels = DATASET_ROOT / split / "labels"
    print(f"{split}: images={images.exists()}, labels={labels.exists()}")

assert (DATASET_ROOT / "data.yaml").is_file(), DATASET_ROOT
assert (DATASET_ROOT / "train" / "images").is_dir()
assert (DATASET_ROOT / "valid" / "images").is_dir()
assert (DATASET_ROOT / "test" / "images").is_dir()

data = yaml.safe_load((DATASET_ROOT / "data.yaml").read_text())
data["path"] = str(DATASET_ROOT)
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

KAGGLE_DATA_YAML = Path("/kaggle/working/chicken-threat-v5-kaggle.yaml")
KAGGLE_DATA_YAML.write_text(yaml.safe_dump(data, sort_keys=False))

DATASET_MANIFEST = str(DATASET_ROOT / "dataset_manifest.json")
print(KAGGLE_DATA_YAML.read_text())

train: images=True, labels=True
valid: images=True, labels=True
test: images=True, labels=True
path: /kaggle/input/datasets/dumdart/chicken-thread-detector-v5-0-0
train: train/images
val: valid/images
test: test/images
names:
- chicken
- other_poultry
- rodent
- fox
- cat
- dog
- bird_of_prey
- other_bird
- person



In [4]:
!git clone $REPOSITORY_URL SmartHomeBridge
%cd SmartHomeBridge
!git checkout $COMMIT_SHA
%pip install -q uv
!uv sync --extra ml --locked

Cloning into 'SmartHomeBridge'...
remote: Enumerating objects: 1207, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 1207 (delta 105), reused 210 (delta 72), pack-reused 938 (from 2)
Receiving objects: 100% (1207/1207), 138.74 MiB | 41.47 MiB/s, done.
Resolving deltas: 100% (622/622), done.
/kaggle/working/SmartHomeBridge
Note: switching to '1bf2b02ed23f00511346a2041330acb39ec9e53a'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 1bf2b02 Me

In [5]:
%env MPLBACKEND=Agg

env: MPLBACKEND=Agg


In [6]:
!MPLBACKEND=Agg uv run smart-home-ml-train --dataset-yaml $KAGGLE_DATA_YAML --training-config ml/chicken_threat/configs/training.yaml --output-dir $OUTPUT_DIR

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.95 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.185 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/chicken-threat-v5-kaggle.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction

In [7]:
BEST_WEIGHTS = f"{OUTPUT_DIR}/chicken-threat-yolo11m-v5/weights/best.pt"

In [8]:
!uv run smart-home-ml-evaluate --weights $BEST_WEIGHTS --dataset-yaml $V4_TEST_YAML --class-mapping ml/chicken_threat/configs/class_mapping.yaml --output $OUTPUT_DIR/v4_test.json --evaluation-name v4_test


Traceback (most recent call last):
  File "/kaggle/working/SmartHomeBridge/.venv/bin/smart-home-ml-evaluate", line 10, in <module>
    sys.exit(evaluate_main())
             ^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src/smart_home_ml/chicken_threat/cli.py", line 147, in evaluate_main
    print(evaluate_model(args.weights, args.dataset_yaml, args.class_mapping, args.output, args.evaluation_name, args.split))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src/smart_home_ml/chicken_threat/artifacts.py", line 58, in evaluate_model
    names = _dataset_class_names(dataset_yaml)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src/smart_home_ml/chicken_threat/artifacts.py", line 204, in _dataset_class_names
    values = _load_yaml(dataset_yaml)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src

In [9]:
!uv run smart-home-ml-evaluate --weights $BEST_WEIGHTS --dataset-yaml $BARN_HOLDOUT_YAML --class-mapping ml/chicken_threat/configs/class_mapping.yaml --output $OUTPUT_DIR/barn_holdout.json --evaluation-name barn_holdout

Traceback (most recent call last):
  File "/kaggle/working/SmartHomeBridge/.venv/bin/smart-home-ml-evaluate", line 10, in <module>
    sys.exit(evaluate_main())
             ^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src/smart_home_ml/chicken_threat/cli.py", line 147, in evaluate_main
    print(evaluate_model(args.weights, args.dataset_yaml, args.class_mapping, args.output, args.evaluation_name, args.split))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src/smart_home_ml/chicken_threat/artifacts.py", line 58, in evaluate_model
    names = _dataset_class_names(dataset_yaml)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src/smart_home_ml/chicken_threat/artifacts.py", line 204, in _dataset_class_names
    values = _load_yaml(dataset_yaml)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src

In [10]:

!uv run smart-home-ml-package-candidate --weights $BEST_WEIGHTS --dataset-manifest $DATASET_MANIFEST --class-mapping ml/chicken_threat/configs/class_mapping.yaml --evaluation $OUTPUT_DIR/v4_test.json --evaluation $OUTPUT_DIR/barn_holdout.json --training-config ml/chicken_threat/configs/training.yaml --output-dir $OUTPUT_DIR --model-id $MODEL_ID

Traceback (most recent call last):
  File "/kaggle/working/SmartHomeBridge/.venv/bin/smart-home-ml-package-candidate", line 10, in <module>
    sys.exit(package_candidate_main())
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src/smart_home_ml/chicken_threat/cli.py", line 160, in package_candidate_main
    print(package_candidate(args.weights, args.dataset_manifest, args.class_mapping, args.evaluation, args.output_dir, args.model_id, args.training_config))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src/smart_home_ml/chicken_threat/artifacts.py", line 99, in package_candidate
    evaluations = [_load_json(path) for path in evaluation_paths]
                   ^^^^^^^^^^^^^^^^
  File "/kaggle/working/SmartHomeBridge/src/smart_home_ml/chicken_threat/artifacts.py", line 246, in _load_json
    return json.load

Evaluate the resulting `best.pt` on both supplied fixed inputs with `smart-home-ml-evaluate`, then package the candidate. Candidate selection and promotion remain manual.